# Figures of NGC 6383's paper by Pulgar-Escobar et al. 2024.

## Module import

In [ ]:
from COSMIC_aux import *

In [ ]:
from astropy.visualization import quantity_support
from astropy.table import QTable, join
from astropy.io import ascii, fits
from astropy.wcs import WCS
from astropy.stats import sigma_clip
import asteca
from sklearn.utils.extmath import weighted_mode
from astropy.stats import knuth_bin_width
from scipy import stats as stats_sc
from datetime import datetime
from astropy.stats import histogram as ashist
%config InlineBackend.figure_format ='retina'
quantity_support();

In [ ]:
## Read of the data and labelling
data_gaia = QTable.read('40_arcmin_clustered.ecsv', guess=False, format='ascii.ecsv')

In [ ]:
# Read the FITS file
with fits.open('../NGC6383_DSS2-red.fits') as hdulist:
    # Extract the data and header information
    data = hdulist[0].data
    header = hdulist[0].header
wcs = WCS(header);

In [ ]:
cluster = (data_gaia['cluster'] == 0) & (data_gaia['probability'] >= 0.5)
noise = (data_gaia['cluster'] == -1) | ((data_gaia['cluster'] == 0) & (data_gaia['probability'] <= 0.5))

In [ ]:
for i in np.unique(data_gaia['cluster']):
    print(f"There's {len(data_gaia[data_gaia['cluster'] == i])} sources in the cluster {i}")

## Parallax clipping

In [ ]:
print(rf'{len(data_gaia[cluster])} before $\sigma$ clipping')
parmax, min_plx, max_plx = sigma_clip(data_gaia['parallax'][cluster], sigma=2, cenfunc=histogram_mode, stdfunc='std',return_bounds=True)
data_gaia['cluster'][(data_gaia['cluster'] == 0) & ((data_gaia['parallax'] < min_plx) | (data_gaia['parallax'] > max_plx))] = -1
cluster = (data_gaia['cluster'] == 0) & (data_gaia['probability'] >= 0.5)
noise = (data_gaia['cluster'] == -1) | ((data_gaia['cluster'] == 0) & (data_gaia['probability'] <= 0.5))
print(rf'{len(data_gaia[cluster])} after $\sigma$ clipping')

In [ ]:
min_plx, max_plx

In [ ]:
fig_prob, ax_prob = plt.subplots(1, 1, figsize=(7, 6), layout='tight')
sc_cmap = ax_prob.scatter(data_gaia['Gmag'][cluster], data_gaia['probability'][cluster], s=9, label='NGC 6383 potential members', c=data_gaia[cluster]['fidelity_v2'], cmap='coolwarm')

cbar = fig_prob.colorbar(sc_cmap, pad=0.01)
cbar.set_label('Astrometric fidelity', fontsize=15)
cbar.ax.tick_params(labelsize=14)  # Set the colorbar tick labels size

ax_prob.axhline(0.6, color='k', ls='--', label=r'$60\%$ of probability')
ax_prob.set_xlabel(r'$G_{{mag}}\,[{}]$'.format(data_gaia['Gmag'].unit), fontsize=16)
ax_prob.set_ylabel(r'Probability', fontsize=16)
ax_prob.legend(loc='lower left', fontsize=10, framealpha=0.4)

ax_prob.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_prob.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_prob.tick_params(axis='both', which='both', direction='in', labelsize=14)

fig_prob.align_labels()
fig_prob.savefig('../Tex_File/Figures/probabilies_post_sigmaclip.pdf', dpi='figure', bbox_inches='tight')

In [ ]:
prob_thresholds = [0.5, 0.6, 0.7, 0.8]

for i in prob_thresholds:
    condition = data_gaia[cluster]['probability'] >= i 
    condition_mag = (data_gaia[cluster]['probability'] >= i) & (data_gaia[cluster]['Gmag'] <= 19*u.mag)
    print(len(data_gaia[cluster][condition]),len(data_gaia[cluster][condition_mag]))

In [ ]:
cluster_data = data_gaia[(data_gaia['cluster'] == 0) & (data_gaia['probability'] >= 0.6)]

## CMD

In [ ]:
#sagitta_data = cluster_data['source_id','parallax','l','b','Gmag','G_BPmag','G_RPmag','j_m','h_m','ks_m',
#                            'parallax_error','e_Gmag','e_G_BPmag','e_G_RPmag','j_msigcom','h_msigcom','ks_msigcom']
#sagitta_data.rename_columns(sagitta_data.colnames,['source_id','parallax','l','b','g','bp','rp','j','h','k','eparallax','eg','ebp','erp','ej','eh','ek'])
#sagitta_data.write('Sagitta/NGC_6383_sagitta.fits',overwrite=True,format='fits')
#!sagitta Sagitta/NGC_6383_sagitta.fits --av_out av_sagitta --pms_out pms_sagitta

In [ ]:
pms = QTable.read('NGC_6383_sagitta-sagitta.fits')
for i in ['av_sagitta','pms_sagitta','age']:
    pms[i] = np.squeeze(pms[i])
# Check if the columns exist in cluster_data
if not any(column in cluster_data.colnames for column in ['av_sagitta', 'pms_sagitta', 'age']):
    # Perform the join operation
    cluster_data = join(cluster_data, pms['source_id', 'av_sagitta', 'pms_sagitta', 'age'], keys='source_id', join_type='left')
    # Replace NaN values in 'pms_sagitta' with 0
    cluster_data['pms_sagitta'] = np.nan_to_num(cluster_data['pms_sagitta'], nan=0)

In [ ]:
# Adding conditions for pms_high and pms_low with non-NaN 'tmass_oid'
pms = (cluster_data['pms_sagitta'] >= 0.6) & ~np.isnan(cluster_data['tmass_oid'])

nopms = (cluster_data['pms_sagitta'] < 0.6) & ~np.isnan(cluster_data['tmass_oid'])

tmass_nan = np.isnan(cluster_data['tmass_oid'])

In [ ]:
# Define bin edges with an interval of 0.1
bins_pms_sagitta = np.arange(0, 1.1, 0.1)  # For PMS Probability (assuming it's in the range [0, 1])
bins_age = np.arange(np.floor(cluster_data['age'].min()), np.ceil(cluster_data['age'].max()) + 0.1, 0.1)  # For log(Age)
bins_av_sagitta = np.arange(np.floor(cluster_data['av_sagitta'].min()), np.ceil(cluster_data['av_sagitta'].max()) + 0.1, 0.1)  # For A_V

# Create the subplots
fig, ax = plt.subplots(3, 1, layout='tight', figsize=(7,7))

# Plot the histograms for 'pms_sagitta'
ax[0].hist(cluster_data['pms_sagitta'], bins=bins_pms_sagitta, color='gray', histtype='step', label='All Data', alpha=0.8)
ax[0].hist(cluster_data[pms]['pms_sagitta'], bins=bins_pms_sagitta, color='orange', histtype='step', label='PMS', alpha=0.85, linestyle='-')
ax[0].hist(cluster_data[nopms]['pms_sagitta'], bins=bins_pms_sagitta, color='blue', histtype='step', label='Non-PMS', alpha=0.85, linestyle='--')
ax[0].hist(cluster_data[tmass_nan]['pms_sagitta'], bins=bins_pms_sagitta, color='green', histtype='step', label='No 2MASS Info', alpha=0.85, linestyle=':')

# Plot the histograms for 'age'
ax[1].hist(cluster_data['age'], bins=bins_age, color='gray', histtype='step', label='All Data', alpha=0.85)
ax[1].hist(cluster_data[pms]['age'], bins=bins_age, color='orange', histtype='step', label='PMS', alpha=0.85, linestyle='-')
ax[1].hist(cluster_data[nopms]['age'], bins=bins_age, color='blue', histtype='step', label='Non-PMS', alpha=0.85, linestyle='--')
ax[1].hist(cluster_data[tmass_nan]['age'], bins=bins_age, color='green', histtype='step', label='No 2MASS Info', alpha=0.85, linestyle=':')

# Plot the histograms for 'av_sagitta'
ax[2].hist(cluster_data['av_sagitta'], bins=bins_av_sagitta, color='gray', histtype='step', label='All Data', alpha=0.85)
ax[2].hist(cluster_data[pms]['av_sagitta'], bins=bins_av_sagitta, color='orange', histtype='step', label='PMS', alpha=0.85, linestyle='-')
ax[2].hist(cluster_data[nopms]['av_sagitta'], bins=bins_av_sagitta, color='blue', histtype='step', label='Non-PMS', alpha=0.85, linestyle='--')
ax[2].hist(cluster_data[tmass_nan]['av_sagitta'], bins=bins_av_sagitta, color='green', histtype='step', label='No 2MASS Info', alpha=0.85, linestyle=':')

# Set the x-axis labels
ax[0].set_xlabel('PMS Probability', fontsize=16)
ax[1].set_xlabel('log(Age)', fontsize=16)
ax[2].set_xlabel(r'$A_V$', fontsize=16)

ax[1].set_xlim([6.05,7.8])
ax[2].set_xlim([1.1,2.2])

# Set the y-axis labels and adjust ticks
for i in ax.flatten():
    i.set_ylabel('Count', fontsize=16)
    i.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    i.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    i.tick_params(axis='both', which='both', direction='in', labelsize=14)
ax[1].legend(fontsize=12)

# Align labels and save the figure
fig.align_labels()
fig.savefig('../Tex_File/Figures/pms_stats.pdf', dpi='figure', bbox_inches='tight')
plt.show()

## ASteCA

In [ ]:
isochs = asteca.isochrones(isochs_path="../MIST/",
                           model='MIST',
                           magnitude="Gaia_G_EDR3",
                           magnitude_effl=6390.7,
                           color=("Gaia_BP_EDR3","Gaia_RP_EDR3"),
                           color_effl = (5182.6,7825.1),
                           color2 = ("Gaia_RP_EDR3","2MASS_J"),
                           color2_effl = (7825.1, 12375.60),
                          )

In [ ]:
synthcl = asteca.synthetic(isochs)
my_cluster = asteca.cluster(
    obs_df=cluster_data.to_pandas(),
    magnitude="Gmag",
    e_mag="e_Gmag",
    color="BP_RP",
    e_color='e_BP_RP',
    ra='ra',
    dec = 'dec',
    plx = 'parallax',
    e_plx = 'parallax_error',
    pmra = 'pmra',
    pmde ='pmdec',
    e_pmra='pmra_error',
    e_pmde='pmdec_error',
    color2 = 'RP_J',
    e_color2 = 'e_RP_J'
)
fix_params = {"alpha": 0.09, 'beta' : 0.94,"Rv": 3.1, "DR":0}
synthcl.calibrate(my_cluster,fix_params)

In [ ]:
import pytensor.tensor as pt

class SyntheticCluster(pt.Op):
    itypes = [pt.dscalar, pt.dscalar, pt.dscalar, pt.dscalar]
    otypes = [pt.dmatrix]

    def perform(self, node, inputs, outputs):
        met, loga, dm, Av = inputs
        params = {"met": met, "loga": loga, "dm": dm, "Av": Av}
        output = synthcl.generate(params, plot_flag=False)
        outputs[0][0] = np.column_stack((output[1], output[0], output[2]))

met_min, met_max = isochs.met_age_dict['met'][0], isochs.met_age_dict['met'][-1]
loga_min, loga_max = isochs.met_age_dict['loga'][0], isochs.met_age_dict['loga'][-1]

with pm.Model() as model:
    dm = pm.TruncatedNormal('dm', mu=10.2, sigma=0.2, lower=10, upper=10.5)
    loga = pm.Uniform('loga', lower=6, upper=7)
    Av = pm.Uniform('Av', lower=0.5, upper=2)
    met = pm.Uniform('met', lower=met_min, upper=met_max)

    synthetic_data = SyntheticCluster()(met, loga, dm, Av)

    observed_data = np.column_stack((cluster_data['BP_RP'].value, cluster_data['Gmag'].value, cluster_data['RP_J'].value))

    sigma = pm.HalfNormal('sigma', sigma=20)
    likelihood = pm.Normal('likelihood', mu=synthetic_data, sigma=sigma, observed=observed_data)

In [ ]:
with model:
    trace = pm.sample(1000,tune=1500,chains=300,step = pm.DEMetropolisZ(),compute_convergence_checks=False)

In [ ]:
store_trace_results(trace,save_trace=True)

In [ ]:
results, trace_loaded = load_results(load_trace=True,only_last=True)

In [ ]:
#specific_datetime = '2024-07-26 02:32:57'
#selected_entry = results[results['Date_Time'] == specific_datetime]
selected_entry = results[results['Date_Time'] == results['Date_Time'].max()]

# Convert the last added rows back into dictionaries
fit_params_mean = selected_entry.set_index('Parameter')['Mean'].to_dict()
fit_params_median = selected_entry.set_index('Parameter')['Median'].to_dict()
fit_params_mode = selected_entry.set_index('Parameter')['Mode'].to_dict()
fit_params_stds = selected_entry.set_index('Parameter')['Std'].to_dict()

# Print loaded dictionaries with the selected data
print(fit_params_mean)
print(fit_params_median)
print(fit_params_mode)
print(fit_params_stds)

In [ ]:
az.summary(trace_loaded,var_names=['~likelihood','~likelihood_unobserved'])

In [ ]:
fig, ax = plt.subplots(5,5,layout='tight',figsize=(15,15))
az.plot_pair(trace_loaded,backend_kwargs = {'layout': 'tight'},kind='hexbin',var_names=['~likelihood','~likelihood_unobserved']
             ,marginals=True,ax=ax,point_estimate='mode',textsize=18,)
plt.show()
fig.savefig('../Tex_File/Figures/plot_pair_trace.pdf',bbox_inches='tight',dpi=600);

In [ ]:
# Adding conditions for pms_high and pms_low with non-NaN 'tmass_oid'
pms_high = pms & (cluster_data['probability'] >= 0.8)
pms_low = pms & (cluster_data['probability'] < 0.8)

nopms_high = nopms & (cluster_data['probability'] >= 0.8)
nopms_low = nopms & (cluster_data['probability'] < 0.8)

high_tmass_nan = tmass_nan & (cluster_data['probability'] >= 0.8)
low_tmass_nan = tmass_nan & (cluster_data['probability'] < 0.8)

In [ ]:
fig_cmd, ax_cmd = plt.subplots(1,1, layout='tight', figsize=(7,7))
best_isochrone_median = synthcl.generate(fit_params_median,plot_flag=True)
best_isochrone_mean = synthcl.generate(fit_params_mean,plot_flag=True)
best_isochrone_mode = synthcl.generate(fit_params_mode,plot_flag=True)

ax_cmd.scatter(cluster_data['BP_RP'][pms_high],cluster_data['Gmag'][pms_high],s=80,alpha=0.95,lw=0,marker='*',c='red',label='PMS Members',zorder=5)
ax_cmd.scatter(cluster_data['BP_RP'][pms_low],cluster_data['Gmag'][pms_low],s=35,alpha=0.85,lw=0,marker='d',c='orangered',label='PMS Probable Members',zorder=4)
ax_cmd.scatter(cluster_data['BP_RP'][nopms_high],cluster_data['Gmag'][nopms_high],s=80,alpha=0.95,lw=0,marker='*',c='blue',label='Non-PMS Members',zorder=5)
ax_cmd.scatter(cluster_data['BP_RP'][nopms_low],cluster_data['Gmag'][nopms_low],s=35,alpha=0.85,lw=0,marker='d',c='blueviolet',label='Non-PMS Probable Members',zorder=4)
ax_cmd.scatter(cluster_data['BP_RP'][high_tmass_nan],cluster_data['Gmag'][high_tmass_nan],s=80,alpha=0.95,lw=0,marker='*',c='k',label='Members',zorder=5)
ax_cmd.scatter(cluster_data['BP_RP'][low_tmass_nan],cluster_data['Gmag'][low_tmass_nan],s=35,alpha=0.85,lw=0,marker='d',c='k',label='Probable Members',zorder=4)
ax_cmd.plot(best_isochrone_mean[1],best_isochrone_mean[0],color='k',zorder=6,label=r'Best mean fit + $\mathcal{N}(\mu,\sigma)$',linestyle='--')
ax_cmd.plot(best_isochrone_median[1],best_isochrone_median[0],color='k',zorder=6,label='Best median fit',linestyle=':')
ax_cmd.plot(best_isochrone_mode[1],best_isochrone_mode[0],color='k',zorder=6,label='Best mode fit',linestyle=(5, (10, 3)))
_, magbins = knuth_bin_width(cluster_data['Gmag'], return_bins=True)
_, colorbins = knuth_bin_width(cluster_data['BP_RP'], return_bins=True)
isochrones_masses = plot_hist2d(synthcl,fit_params_mode, fit_params_stds, ax_cmd,n_samples=1000,cmin=2000,alpha=0.3,bins=[colorbins,magbins],return_masses=True)
ax_cmd.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_cmd.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_cmd.tick_params(axis='both', which='both', direction='in',labelsize=14)
ax_cmd.set(xlim=[-0.029,1.02*np.max(cluster_data['BP_RP'])],ylim=[0.99*np.min(cluster_data['Gmag']),1.01*np.max(cluster_data['Gmag'])])
ax_cmd.legend()
ax_cmd.set_xlabel(r'Color $(G_{{BP}} - G_{{RP}})$ [{}]'.format(cluster_data['BP_RP'].unit),fontsize=16)
ax_cmd.set_ylabel(r'$G_{{mag}}\,[{}]$'.format(cluster_data['Gmag'].unit),fontsize=16)
#ax_cmd.annotate("", xy=(x0_gaia + dx_gaia, y0_gaia + dy_gaia), xytext=(x0_gaia, y0_gaia), arrowprops=prop)
plot_errors_bar(cluster_data['Gmag'],cluster_data['BP_RP'], cluster_data['e_Gmag'], cluster_data['e_BP_RP'], ax_cmd,loc=0.1*u.mag)
ax_cmd.invert_yaxis()
plt.show()
#fig_cmd.savefig('../Tex_File/Figures/ngc6383_cmd.pdf',dpi=1500);

In [ ]:
fig_multicmd, ax_multicmd = plt.subplots(1,3, layout='tight', figsize=(15,7))

ax_multicmd[0].scatter(cluster_data['BP_RP'][pms_high],cluster_data['Gmag'][pms_high],s=80,alpha=0.95,lw=0,marker='*',c='red',label='PMS Members',zorder=5)
ax_multicmd[0].scatter(cluster_data['BP_RP'][pms_low],cluster_data['Gmag'][pms_low],s=35,alpha=0.85,lw=0,marker='d',c='orangered',label='PMS Probable Members',zorder=4)
ax_multicmd[0].scatter(cluster_data['BP_RP'][nopms_high],cluster_data['Gmag'][nopms_high],s=80,alpha=0.95,lw=0,marker='*',c='blue',label='Non-PMS Members',zorder=5)
ax_multicmd[0].scatter(cluster_data['BP_RP'][nopms_low],cluster_data['Gmag'][nopms_low],s=35,alpha=0.85,lw=0,marker='d',c='blueviolet',label='Non-PMS Probable Members',zorder=4)
ax_multicmd[0].scatter(cluster_data['BP_RP'][high_tmass_nan],cluster_data['Gmag'][high_tmass_nan],s=80,alpha=0.95,lw=0,marker='*',c='k',label='Members',zorder=5)
ax_multicmd[0].scatter(cluster_data['BP_RP'][low_tmass_nan],cluster_data['Gmag'][low_tmass_nan],s=35,alpha=0.85,lw=0,marker='d',c='k',label='Probable Members',zorder=4)
ax_multicmd[0].set_xlabel(r'Color $(G_{{BP}} - G_{{RP}})$ [{}]'.format(cluster_data['BP_RP'].unit),fontsize=16)
ax_multicmd[0].set_ylabel(r'$G_{{mag}}\,[{}]$'.format(cluster_data['Gmag'].unit),fontsize=16)
ax_multicmd[1].scatter(cluster_data['RP_J'][pms_high],cluster_data['Gmag'][pms_high],s=80,alpha=0.95,lw=0,marker='*',c='red',label='PMS Members',zorder=5)
ax_multicmd[1].scatter(cluster_data['RP_J'][pms_low],cluster_data['Gmag'][pms_low],s=35,alpha=0.85,lw=0,marker='d',c='orangered',label='PMS Probable Members',zorder=4)
ax_multicmd[1].scatter(cluster_data['RP_J'][nopms_high],cluster_data['Gmag'][nopms_high],s=80,alpha=0.95,lw=0,marker='*',c='blue',label='Non-PMS Members',zorder=5)
ax_multicmd[1].scatter(cluster_data['RP_J'][nopms_low],cluster_data['Gmag'][nopms_low],s=35,alpha=0.85,lw=0,marker='d',c='blueviolet',label='Non-PMS Probable Members',zorder=4)
ax_multicmd[1].set_xlabel(r'Color $(G_{{RP}} - J)$ [{}]'.format(cluster_data['BP_RP'].unit),fontsize=16)
ax_multicmd[1].set_ylabel(r'$G_{{mag}}\,[{}]$'.format(cluster_data['Gmag'].unit),fontsize=16)
#ax_multicmd[0].annotate("", xy=(x0 + dx, y0 + dy), xytext=(x0, y0), arrowprops=prop)

logAges = np.arange(6.2, 7.1, 0.2)  # From 6.1 to 7.0, inclusive
num_colors = len(logAges)
unique_color_indices = np.linspace(0, 1, num_colors, endpoint=False)
colors = [plt.cm.hsv(x) for x in unique_color_indices[::-1]]
for i, logAge in enumerate(logAges):
    params = fit_params_mode.copy()
    params['loga'] = round(logAge,2)
    isochrones_synth = synthcl.generate(params,plot_flag=True)
    cut_isochrones = (isochrones_synth[0] >= np.nanmin(cluster_data['Gmag'].value))
    isochrones_synth = isochrones_synth[:,cut_isochrones]
    ax_multicmd[0].plot(isochrones_synth[1],isochrones_synth[0],lw=2,label=fr'$\log_{{\mathrm{{age}}}} = {np.round(logAge,2)}$')
    ax_multicmd[1].plot(isochrones_synth[2],isochrones_synth[0],lw=2,label=fr'$\log_{{\mathrm{{age}}}} = {np.round(logAge,2)}$')

# Swap x and y data in scatter plots
ax_multicmd[2].scatter(cluster_data['BP_RP'][pms_high], cluster_data['RP_J'][pms_high], s=80, alpha=0.95, lw=0, marker='*', c='red', label='PMS Members', zorder=5)
ax_multicmd[2].scatter(cluster_data['BP_RP'][pms_low], cluster_data['RP_J'][pms_low], s=35, alpha=0.85, lw=0, marker='d', c='orangered', label='PMS Probable Members', zorder=4)
ax_multicmd[2].scatter(cluster_data['BP_RP'][nopms_high], cluster_data['RP_J'][nopms_high], s=80, alpha=0.95, lw=0, marker='*', c='blue', label='Non-PMS Members', zorder=5)
ax_multicmd[2].scatter(cluster_data['BP_RP'][nopms_low], cluster_data['RP_J'][nopms_low], s=35, alpha=0.85, lw=0, marker='d', c='blueviolet', label='Non-PMS Probable Members', zorder=4)
ax_multicmd[2].set_xlabel(r'Color $(G_{{RP}} - G_{{RP}})$ [{}]'.format(cluster_data['BP_RP'].unit), fontsize=16)
ax_multicmd[2].set_ylabel(r'Color $(G_{{RP}} - J)$ [{}]'.format(cluster_data['BP_RP'].unit),fontsize=16)
# Swap x and y data in isochrones plot
for i, logAge in enumerate(logAges):
    params = fit_params_mode.copy()
    params['loga'] = logAge
    isochrones_synth = synthcl.generate(params,plot_flag=True)
    ax_multicmd[2].plot(isochrones_synth[1], isochrones_synth[2], lw=2, label=fr'$\log_{{\mathrm{{age}}}} = {np.round(logAge,2)}$')
    
ax_multicmd[0].scatter([],[],label=rf'$Z_{{ini}}={fit_params_mode['met']}$',s=0)
ax_multicmd[1].scatter([],[],label=rf'$Z_{{ini}}={fit_params_mode['met']}$',s=0)
ax_multicmd[2].scatter([], [], label=rf'$Z_{{ini}}={fit_params_mode['met']}$', s=0)  # Placeholder for Zini label

ax_multicmd[0].set(xlim=[0.6*np.nanmin(cluster_data['BP_RP']),1.2*np.nanmax(cluster_data['BP_RP'])],ylim=[0.98*np.nanmin(cluster_data['Gmag']),1.01*np.max(cluster_data['Gmag'])])
ax_multicmd[1].set(xlim=[0.6*np.nanmin(cluster_data['RP_J']),1.2*np.nanmax(cluster_data['RP_J'])],ylim=[0.98*np.nanmin(cluster_data['Gmag']),1.01*np.max(cluster_data['Gmag'])])
for i, ax in enumerate(ax_multicmd.flatten()):
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.tick_params(axis='both', which='both', direction='in',labelsize=14) 
    ax.legend(framealpha=0.8)
    if i != 2:
        ax.invert_yaxis()
#fig_multicmd.savefig('../Tex_File/Figures/ngc6383_cmd_various.pdf',dpi=1500);

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Configuración de subplots para una cuadrícula de 3x3
fig_error, ax_error = plt.subplots(3, 3, figsize=(20, 8), layout='tight')

# Gráfico del error en G_BP contra G_BP
mean_g_bp_full = np.nanmean(data_gaia['e_G_BPmag'])
median_g_bp_full = np.nanmedian(data_gaia['e_G_BPmag'])
ax_error[0, 0].scatter(data_gaia['G_BPmag'], data_gaia['e_G_BPmag'], s=10, alpha=0.3, c='blue', label='Full sample')
ax_error[0, 0].scatter(cluster_data['G_BPmag'], cluster_data['e_G_BPmag'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[0, 0].axhline(mean_g_bp_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_g_bp_full:.4f}', zorder=10)
ax_error[0, 0].axhline(median_g_bp_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_g_bp_full:.4f}', zorder=10)
ax_error[0, 0].set_xlabel(r'$G_{{BP}} \, [mag]$', fontsize=16)
ax_error[0, 0].set_ylabel(r'$\sigma_{{BP}} \, [mag]$', fontsize=16)

# Gráfico del error en G_RP contra G_RP
mean_g_rp_full = np.nanmean(data_gaia['e_G_RPmag'])
median_g_rp_full = np.nanmedian(data_gaia['e_G_RPmag'])
ax_error[1, 0].scatter(data_gaia['G_RPmag'], data_gaia['e_G_RPmag'], s=10, alpha=0.3, c='red', label='Full sample')
ax_error[1, 0].scatter(cluster_data['G_RPmag'], cluster_data['e_G_RPmag'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[1, 0].axhline(mean_g_rp_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_g_rp_full:.4f}', zorder=10)
ax_error[1, 0].axhline(median_g_rp_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_g_rp_full:.4f}', zorder=10)
ax_error[1, 0].set_xlabel(r'$G_{{RP}} \, [mag]$', fontsize=16)
ax_error[1, 0].set_ylabel(r'$\sigma_{{RP}} \, [mag]$', fontsize=16)

# Gráfico del error en G contra G
mean_g_full = np.nanmean(data_gaia['e_Gmag'])
median_g_full = np.nanmedian(data_gaia['e_Gmag'])
ax_error[2, 0].scatter(data_gaia['Gmag'], data_gaia['e_Gmag'], s=10, alpha=0.3, c='green', label='Full sample')
ax_error[2, 0].scatter(cluster_data['Gmag'], cluster_data['e_Gmag'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[2, 0].axhline(mean_g_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_g_full:.4f}', zorder=10)
ax_error[2, 0].axhline(median_g_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_g_full:.4f}', zorder=10)
ax_error[2, 0].set_xlabel(r'$G \, [mag]$', fontsize=16)
ax_error[2, 0].set_ylabel(r'$\sigma_G \, [mag]$', fontsize=16)

# Additional plot 1: Error in J vs J magnitude (data_gaia)
mean_j_full = np.nanmean(data_gaia['j_msigcom'])
median_j_full = np.nanmedian(data_gaia['j_msigcom'])
ax_error[0, 1].scatter(data_gaia['j_m'], data_gaia['j_msigcom'], s=10, alpha=0.3, c='purple', label='Full sample')
ax_error[0, 1].scatter(cluster_data['j_m'], cluster_data['j_msigcom'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[0, 1].axhline(mean_j_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_j_full:.4f}', zorder=10)
ax_error[0, 1].axhline(median_j_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_j_full:.4f}', zorder=10)
ax_error[0, 1].set_xlabel(r'$J \, [mag]$', fontsize=16)
ax_error[0, 1].set_ylabel(r'$\sigma_J \, [mag]$', fontsize=16)

# Additional plot 2: Error in H vs H magnitude (data_gaia)
mean_h_full = np.nanmean(data_gaia['h_msigcom'])
median_h_full = np.nanmedian(data_gaia['h_msigcom'])
ax_error[1, 1].scatter(data_gaia['h_m'], data_gaia['h_msigcom'], s=10, alpha=0.3, c='magenta', label='Full sample')
ax_error[1, 1].scatter(cluster_data['h_m'], cluster_data['h_msigcom'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[1, 1].axhline(mean_h_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_h_full:.4f}', zorder=10)
ax_error[1, 1].axhline(median_h_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_h_full:.4f}', zorder=10)
ax_error[1, 1].set_xlabel(r'$H \, [mag]$', fontsize=16)
ax_error[1, 1].set_ylabel(r'$\sigma_H \, [mag]$', fontsize=16)

# Additional plot 3: Error in K vs K magnitude (data_gaia)
mean_k_full = np.nanmean(data_gaia['ks_msigcom'])
median_k_full = np.nanmedian(data_gaia['ks_msigcom'])
ax_error[2, 1].scatter(data_gaia['ks_m'], data_gaia['ks_msigcom'], s=10, alpha=0.3, c='brown', label='Full sample')
ax_error[2, 1].scatter(cluster_data['ks_m'], cluster_data['ks_msigcom'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[2, 1].axhline(mean_k_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_k_full:.4f}', zorder=10)
ax_error[2, 1].axhline(median_k_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_k_full:.4f}', zorder=10)
ax_error[2, 1].set_xlabel(r'$K_s \, [mag]$', fontsize=16)
ax_error[2, 1].set_ylabel(r'$\sigma_{K_s} \, [mag]$', fontsize=16)

# Additional plot 4: Error in W1 vs W1 magnitude (WISE)
mean_w1_full = np.nanmean(data_gaia['w1mpro_error'])
median_w1_full = np.nanmedian(data_gaia['w1mpro_error'])
ax_error[0, 2].scatter(data_gaia['w1mpro'], data_gaia['w1mpro_error'], s=10, alpha=0.3, c='darkgreen', label='Full sample')
ax_error[0, 2].scatter(cluster_data['w1mpro'], cluster_data['w1mpro_error'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[0, 2].axhline(mean_w1_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_w1_full:.4f}', zorder=10)
ax_error[0, 2].axhline(median_w1_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_w1_full:.4f}', zorder=10)
ax_error[0, 2].set_xlabel(r'$W1 \, [mag]$', fontsize=16)
ax_error[0, 2].set_ylabel(r'$\sigma_{W1} \, [mag]$', fontsize=16)

# Additional plot 5: Error in W2 vs W2 magnitude (WISE)
mean_w2_full = np.nanmean(data_gaia['w2mpro_error'])
median_w2_full = np.nanmedian(data_gaia['w2mpro_error'])
ax_error[1, 2].scatter(data_gaia['w2mpro'], data_gaia['w2mpro_error'], s=10, alpha=0.3, c='darkred', label='Full sample')
ax_error[1, 2].scatter(cluster_data['w2mpro'], cluster_data['w2mpro_error'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[1, 2].axhline(mean_w2_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_w2_full:.4f}', zorder=10)
ax_error[1, 2].axhline(median_w2_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_w2_full:.4f}', zorder=10)
ax_error[1, 2].set_xlabel(r'$W2 \, [mag]$', fontsize=16)
ax_error[1, 2].set_ylabel(r'$\sigma_{W2} \, [mag]$', fontsize=16)

# Additional plot 6: Error in W3 vs W3 magnitude (WISE)
mean_w3_full = np.nanmean(data_gaia['w3mpro_error'])
median_w3_full = np.nanmedian(data_gaia['w3mpro_error'])
ax_error[2, 2].scatter(data_gaia['w3mpro'], data_gaia['w3mpro_error'], s=10, alpha=0.3, c='teal', label='Full sample')
ax_error[2, 2].scatter(cluster_data['w3mpro'], cluster_data['w3mpro_error'], s=10, alpha=1, c='orange', label='Cluster data', zorder=5)
ax_error[2, 2].axhline(mean_w3_full, color='black', linestyle='--', linewidth=1.5, label=f'Mean = {mean_w3_full:.4f}', zorder=10)
ax_error[2, 2].axhline(median_w3_full, color='purple', linestyle=':', linewidth=1.5, label=f'Median = {median_w3_full:.4f}', zorder=10)
ax_error[2, 2].set_xlabel(r'$W3 \, [mag]$', fontsize=16)
ax_error[2, 2].set_ylabel(r'$\sigma_{W3} \, [mag]$', fontsize=16)

# Final adjustments for all subplots
for ax in ax_error.flatten():
    ax.tick_params(axis='both', which='both', direction='in', labelsize=14)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.invert_xaxis()
    ax.legend()

# Save or display the figure
fig_error.savefig('../Tex_File/Figures/errors_vs_magnitude.pdf', dpi=1500)
plt.show()

In [ ]:
# Color-color indices for the full sample
w1_w2_full = data_gaia['w1mpro'] - data_gaia['w2mpro']
w2_w3_full = data_gaia['w2mpro'] - data_gaia['w3mpro']
h_k_full = data_gaia['h_m'] - data_gaia['ks_m']

# Color-color indices for the cluster data
w1_w2_cluster = cluster_data['w1mpro'] - cluster_data['w2mpro']
w2_w3_cluster = cluster_data['w2mpro'] - cluster_data['w3mpro']
h_k_cluster = cluster_data['h_m'] - cluster_data['ks_m']

# Plot configuration
fig, axs = plt.subplots(1, 2, figsize=(10, 5), layout='tight')

# Top panel: [3.4 µm−4.6 µm] × [4.6 µm−12 µm]
axs[0].scatter(w1_w2_full, w2_w3_full, s=50, c='blue', alpha=0.3, label='Full sample')
axs[0].scatter(w1_w2_cluster, w2_w3_cluster, s=50, c='red', alpha=0.9, label='Cluster data', zorder=5)
axs[0].set_xlabel('[4.6] - [12] (W2 - W3)', fontsize=14)
axs[0].set_ylabel('[3.4] - [4.6] (W1 - W2)', fontsize=14)

# Finding intersections
# Intersection of w1−w2 >− 0.42 × (w2− w3) + 2.2 and w1−w2 > 0.46 × (w2− w3)− 0.9
intersection_w2_w3 = (2.2 - (-0.9)) / (0.46 + 0.42)
intersection_w1_w2 = -0.42 * intersection_w2_w3 + 2.2

# Class I boundaries:
# Adjust the vertical line to end at the intersection
axs[0].plot([2.0, 2.0], [-0.42 * 2.0 + 2.2, 3], 'k-')  # until intersection
axs[0].plot([4.5, 4.5], [0.46 * 4.5 - 0.9, 3], 'k-')  # until intersection

# Adjust the diagonal lines to intersect correctly
axs[0].plot([2.0, intersection_w2_w3], [-0.42 * 2.0 + 2.2, intersection_w1_w2], 'k-')
axs[0].plot([intersection_w2_w3, 4.5], [intersection_w1_w2, 0.46 * 4.5 - 0.9], 'k-')

# Class II boundaries:

# w1−w2 < 0.9 × (w2− w3)− 0.25
axs[0].plot([2.0, intersection_w2_w3_class2_1], [0.9 * 2.0 - 0.25, intersection_w1_w2_class2_1], 'k--')


# Bottom panel: [3.4 µm−4.6 µm] × [H-K]
axs[1].scatter(w1_w2_full, h_k_full, s=50, c='green', alpha=0.3, label='Full sample')
axs[1].scatter(w1_w2_cluster, h_k_cluster, s=50, c='red', alpha=0.9, label='Cluster data', zorder=5)

axs[1].set_xlabel('[3.4] - [4.6] (W1 - W2)', fontsize=14)
axs[1].set_ylabel('H - K', fontsize=14)

# Adding legends
axs[0].legend()
axs[1].legend()
plt.savefig('../Tex_File/Figures/yso_class.pdf', dpi=800)
plt.show()

## Parallax

In [ ]:
parallax_results = parallax_determination(cluster_data, return_trace=True,savefig='../Tex_File/Figures/',prob_thresholds=[60],paper_single=True)

In [ ]:
parallax_results

## Proper motion

In [ ]:
pm_results, pm_dist, pmprob = pm_determination(cluster_data,return_pmdist=True,return_pmprob=True,savefig='../Tex_File/Figures/',prob_number=[60],paper_single=True,return_trace=True)

In [ ]:
pm_results

## Projected velocity

In [ ]:
cluster_data['projected_velocity'] = np.sqrt(cluster_data['pmra']**2 + cluster_data['pmdec']**2)

In [ ]:
results_vp = velocity_determination(cluster_data, return_trace=True,savefig='../Tex_File/Figures/',prob_thresholds=[60],paper_single=True)

## Center determination

In [ ]:
centers, params_list = graph_center_determination(data=cluster_data,projection=wcs,weighted=True,weight_array=pmprob[0],weight_prob=60,prob_number=None
                                                  ,savefig='../Tex_File/Figures/',only_weight=True,paper_single=True,distance_scale=parallax_results['mu_r_mean'][0]*u.kpc)

In [ ]:
cluster_data['d_center'] = angular_separation(cluster_data['ra'], cluster_data['dec'], centers[0][0][0], centers[0][0][1]).to(u.arcmin)

## Masses

In [ ]:
synthcl.get_models(fit_params_mode, fit_params_stds, [centers[0][0][0].value,centers[0][0][1].value],N_models=1000)

In [ ]:
df_masses_bprob = synthcl.stellar_masses()

In [ ]:
df_masses_bprob['mass'] = np.where(df_masses_bprob['binar_prob'] >= 0.7, 
                                   df_masses_bprob['m1'] + df_masses_bprob['m2'], 
                                   df_masses_bprob['m1'])
masses = QTable.from_pandas(df_masses_bprob)

In [ ]:
masses_dict = synthcl.cluster_masses()

In [ ]:
# Print the median mass values and their STDDEV
for k, arr in masses_dict.items():
    print("{:<8}: {:.0f}+/-{:.0f}".format(k, np.median(arr), np.std(arr)))

In [ ]:
cluster_data.add_columns([masses['m1'],masses['m1_std'],masses['m2'],masses['m2_std'],masses['binar_prob'],masses['mass']])

In [ ]:
cluster_data['m1'].unit = u.Msun
cluster_data['m1_std'].unit = u.Msun
cluster_data['m2'].unit = u.Msun
cluster_data['m2_std'].unit = u.Msun
cluster_data['mass'].unit = u.Msun

In [ ]:
cluster_data['mass_std'] = np.where(df_masses_bprob['binar_prob'] >= 0.7, 
                                   np.sqrt(df_masses_bprob['m1_std']**2 + df_masses_bprob['m2_std']**2), 
                                   df_masses_bprob['m1_std'])

In [ ]:
fig_massbinary, ax_massbinary = plt.subplots(2,1, figsize=(8,14))
best_isochrone_mode = synthcl.generate(fit_params_mode,plot_flag=True)
sc_mass = ax_massbinary[0].scatter(cluster_data['BP_RP'],cluster_data['Gmag'],s=80,lw=0,marker='*',c=cluster_data['mass'].value,zorder=5,cmap='coolwarm_r')
sc_binary = ax_massbinary[1].scatter(cluster_data['BP_RP'],cluster_data['Gmag'],s=80,lw=0,marker='*',c=cluster_data['binar_prob'],zorder=5,cmap='coolwarm_r')
ax_massbinary[0].scatter(cluster_data['BP_RP'],cluster_data['Gmag'],s=80,alpha=0.95,lw=0,marker='*',zorder=-1,c='black')
ax_massbinary[1].scatter(cluster_data['BP_RP'],cluster_data['Gmag'],s=80,alpha=0.95,lw=0,marker='*',zorder=-1,c='black')

cbar1 = fig_massbinary.colorbar(sc_mass,location='right',pad=0.01)
cbar1.set_label(r'Total mass $[M_\odot]$',fontsize=16)
cbar2 = fig_massbinary.colorbar(sc_binary,location='right',pad=0.01)
cbar2.set_label('Binary probability',fontsize=16),
cbar1.ax.tick_params(labelsize=14)
cbar2.ax.tick_params(labelsize=14)
for ax in ax_massbinary.flatten():
    ax.plot(best_isochrone_mode[1],best_isochrone_mode[0],color='k',zorder=-1,label=r'Best mode fit',linestyle='--')
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.set(xlim=[0,1.02*np.max(cluster_data['BP_RP'])],ylim=[0.99*np.min(cluster_data['Gmag']),1.02*np.max(cluster_data['Gmag'])])
    ax.legend()
    ax.set_ylabel(r'$G_{{mag}}\,[{}]$'.format(cluster_data['Gmag'].unit),fontsize=16)
    ax.tick_params(axis='both', which='both', direction='in',labelsize=14)
    ax.invert_yaxis()
ax_massbinary[1].set_xlabel(r'Color $(G_{{BP}} - G_{{RP}})$ [{}]'.format(cluster_data['BP_RP'].unit),fontsize=16)
fig_massbinary.subplots_adjust(hspace=0.01)
fig_massbinary.savefig('../Tex_File/Figures/ngc6383_mass_binary.pdf', bbox_inches='tight')
plt.show()

In [ ]:
prob_thresholds = [0.5, 0.6, 0.7, 0.8]

for i in prob_thresholds:
    condition = cluster_data['probability'] >= i 
    condition_mag = (cluster_data['probability'] >= i) & (cluster_data['Gmag'] <= 18*u.mag)
    print(len(cluster_data[condition]),len(cluster_data[condition_mag]),np.nanmax(cluster_data['mass'][condition]),np.nanmax(cluster_data['mass'][condition_mag]))

In [ ]:
np.nanmean(cluster_data['mass'])

In [ ]:
np.nanmean(cluster_data['m1'])

## Radial Density Profile (cite Andreas H. W. Kupper 2010)

3P King Porfile

$
f(R)  = k \left[\dfrac{1}{\sqrt{1+\left(\frac{R}{R_c}\right)^2}} - \dfrac{1}{\sqrt{1+\left(\frac{R_t}{R_c}\right)^2}}\right]^2+b
$

and 2P King Profile

$
f(R) = \dfrac{k}{1+\left(\frac{R}{R_c}\right)^2}+b
$

### Models

In [ ]:
hunt_mass = 902.27*u.Msun
hunt_mass_err = 92.29*u.Msun

In [ ]:
ra = centers[0][0][0]
dec = centers[0][0][1]
distance = parallax_results['mu_r_mean'][0]*u.kpc
ra_err = centers[0][1][0]
dec_err = centers[0][1][1]
distance_err = parallax_results['mu_parallax_std'][0]*u.kpc

d_gc, d_gc_err = calculate_galactocentric_distance(ra, dec, distance, ra_err, dec_err, distance_err)
print(d_gc,d_gc_err)
# Assume you have the functions calculate_galactocentric_distance, calculate_galactic_mass, estimate_cluster_mass
hill_results = calculate_hill_radius(distance, distance_err, center=[ra,dec],galactocentric_distance=d_gc,galactocentric_distance_err=d_gc_err,cluster_mass=hunt_mass,cluster_mass_err=hunt_mass_err, return_linear_size=True,return_galactic_mass=True)
print(hill_results)

In [ ]:
king_results = graph_king(data=cluster_data,centers=centers,savefig='../Tex_File/Figures/',
                          density_method='equip',distances=parallax_results['mu_r_mean'],
                          distances_err = parallax_results['mu_parallax_std'],
                          return_priors=True,prob_number=[0.6],paper_single=True,
                          cluster_mass=hunt_mass,cluster_mass_err=hunt_mass_err,log_scale=True)

In [ ]:
king_results

In [ ]:
abs_mag = calculate_absolute_magnitude(cluster_data['Gmag'],parallax_results['mu_r_mean']*u.kpc)

min_mag = np.floor(np.nanmin(abs_mag)).value
max_mag = np.ceil(np.nanmax((abs_mag))).value
fig,ax = plt.subplots(2,1,layout='tight',figsize=(7,10))
ax[0].hist(abs_mag,bins=np.arange(min_mag, max_mag + 1, 1),histtype='step',hatch='//',color='orange')
ax[0].set_xlabel(r'$G_\mathrm{abs}$ [mag]',fontsize=16)
ax[0].set_ylabel('Counts',fontsize=16)

min_mag = np.floor(np.min(cluster_data['Gmag'])).value
max_mag = np.ceil(np.max(cluster_data['Gmag'])).value

ax[1].hist(cluster_data['Gmag'],bins=np.arange(min_mag, max_mag + 1, 1),histtype='step',hatch='//',color='orange')
ax[1].set_xlabel(r'$G_\mathrm{mag}$ [mag]',fontsize=16)
ax[1].set_ylabel('Counts',fontsize=16)
ax_cmd.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_cmd.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_cmd.tick_params(axis='both', which='both', direction='in',labelsize=14)

fig.savefig('../Tex_File/Figures/luminosity_function.pdf',bbox_inches='tight')

In [ ]:
from astropy.visualization.wcsaxes import SphericalCircle

In [ ]:
king_results['priors'][0]

In [ ]:
R_t_mean =king_results['bayesian_results']['R_t_mean'][0]
R_c_mean =king_results['bayesian_results']['R_c_mean'][0]
R_h = king_results['bayesian_results']['half_light_radius']
R_hill = king_results['priors'][0]['hill_radius']
R_bound = king_results['priors'][0]['gravitational_bound_radius']
R_hm =half_mass_radius(cluster_data,centers=centers[0][0],distance=parallax_results['mu_parallax_mean'])

In [ ]:
R_h

In [ ]:
print('R_h:',R_h)
print('R_hm:',R_hm)

In [ ]:
# Create the figure and axes with the WCS projectio
fig_real, ax_real = plt.subplots(1, 1,figsize=(12,12), subplot_kw={'projection': wcs})

# Plot the FITS image
cc = ax_real.imshow(data, cmap='rainbow',aspect='equal')
ax_real.scatter(cluster_data['ra'],cluster_data['dec'],color='darkred', transform=ax_real.get_transform('world'),alpha=0.6,s=10,label='NGC 6383 sources',zorder=4)
ax_real.scatter(centers[0][0][0],centers[0][0][1],marker='1', s=400, color='blue', transform=ax_real.get_transform('world'),lw=1,alpha=0.85,label='Center')
from astropy.visualization.wcsaxes import SphericalCircle
import astropy.units as u

# Assuming 'ax_real' is already set up with an appropriate WCS projection
# Define colors for better differentiation
colors = ['red', 'darkslategrey', 'darkblue', 'darkred', 'magenta', 'yellow', 'orangered']
circle_search = SphericalCircle((263.67148711*u.deg, -32.57727207*u.deg), 40 * u.arcmin, edgecolor=colors[0], facecolor='none', transform=ax_real.get_transform('world'), label='Cone Search')
circle_search.set(linestyle='--', alpha=0.85, linewidth=1.6)
# Tidal radius circle
circle_tidal = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_t_mean, edgecolor=colors[1], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_t = {:.2f} \,{}$'.format(R_t_mean.value, u.arcmin))
circle_tidal.set(linestyle='--', alpha=0.85, linewidth=1.6)
# Core radius circle
circle_core = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_c_mean, edgecolor=colors[2], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_c = {:.2f}\,{}$'.format(R_c_mean.value, u.arcmin))
circle_core.set(linestyle='solid', alpha=0.85, linewidth=1.6)
# Half-light radius circle
circle_rh = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_h['R_h'], edgecolor=colors[3], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_{{hl}} = {:.2f}\,{}$'.format(R_h['R_h'].value, u.arcmin))
circle_rh.set(linestyle='-.', alpha=0.85, linewidth=1.6)
# Half-mass radius circle
circle_rm = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_hm[0], edgecolor=colors[4], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_{{hm}} = {:.2f}\,{}$'.format(R_hm[0].value, u.arcmin))
circle_rm.set(linestyle='-', alpha=0.85, linewidth=1.6)
# Hill radius circle
circle_hill = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_hill, edgecolor=colors[5], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_{{hill}} = {:.2f}\,{}$'.format(R_hill.value, u.arcmin))
circle_hill.set(linestyle='dashdot', alpha=0.85, linewidth=1.6)
# Bound radius circle
circle_bound = SphericalCircle((centers[0][0][0], centers[0][0][1]), R_bound, edgecolor=colors[6], facecolor='none', transform=ax_real.get_transform('world'), label=r'$R_{{bound}} = {:.2f}\,{}$'.format(R_bound.value, u.arcmin))
circle_bound.set(linestyle='solid', alpha=0.85, linewidth=1.6)

# Add all patches to the axes
ax_real.add_patch(circle_search),ax_real.add_patch(circle_tidal),ax_real.add_patch(circle_core),ax_real.add_patch(circle_rh),ax_real.add_patch(circle_rm),ax_real.add_patch(circle_hill),ax_real.add_patch(circle_bound)
ax_real.set_autoscale_on(False)

# Add coordinate axes
ax_real.set_xlabel(r'$\alpha$ [{}]'.format(cluster_data['ra'].unit),fontsize=16)
ax_real.set_ylabel(r'$\delta$ [{}]'.format(cluster_data['dec'].unit),fontsize=16)
ax_real.coords[0].set_major_formatter('d.dd')
ax_real.coords[1].set_major_formatter('d.dd')
ax_real.coords.grid(True, color='white', linestyle='dotted', alpha=0.5)
ax_real.xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_real.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_real.tick_params(axis='both', which='both', direction='in',fontsize=14)
ax_real.set_aspect('equal')
gc_distance,scalebar_lenght = 1/(parallax_results['mu_parallax_mean'][0])*u.kpc,5 * u.pc
scalebar_angle = (scalebar_lenght / gc_distance).to(u.deg, equivalencies=u.dimensionless_angles())
add_scalebar(ax_real, scalebar_angle, label="5 pc", color="white",corner='bottom right');
ax_real.legend(loc='upper left')
fig_real.savefig('../Tex_File/Figures/real_sky.pdf',bbox_inches='tight',dpi=800);

In [ ]:
def half_mass_relaxation_time(N, rh, M, sigma_N=0, sigma_rh=0*u.pc, sigma_M=0*u.Msun, lambda_value=0.11):
    """
    Calculate the half-mass relaxation time for a star cluster and its uncertainty.
    
    Parameters:
    N (int): Number of cluster members.
    rh (Quantity): Half-mass radius of the cluster with units of parsecs.
    M (Quantity): Total mass of the cluster with units of solar masses.
    sigma_N (float): Uncertainty in the number of cluster members.
    sigma_rh (Quantity): Uncertainty in the half-mass radius with units of parsecs.
    sigma_M (Quantity): Uncertainty in the total mass with units of solar masses.
    lambda_value (float): The constant from Giersz & Heggie (1994), typically around 0.11.
    
    Returns:
    tuple: The half-mass relaxation time in millions of years and its uncertainty.
    """
    # Calculate the half-mass relaxation time in Myr using the given formula
    t_rh = (0.17 * N / np.log(lambda_value * N)) * np.sqrt(rh**3 / (G * M)).to(u.Myr)

    # Partial derivatives
    d_trh_dN = t_rh / N * (1 + 1 / np.log(lambda_value * N))
    d_trh_drh = 1.5 * t_rh / rh
    d_trh_dM = -0.5 * t_rh / M
    # Propagate the uncertainties
    sigma_trh = np.sqrt((d_trh_dN * sigma_N)**2 + (d_trh_drh * sigma_rh)**2 + (d_trh_dM * sigma_M)**2)

    return t_rh, sigma_trh.to(u.Myr)

t_rh_hunt, sigma_trh_hunt = half_mass_relaxation_time(N=len(cluster_data), rh=R_hm[1][0], M=hunt_mass, sigma_N=0, sigma_rh=R_hm[3][0], sigma_M=hunt_mass_err, lambda_value=0.11)

print(f"Half-mass relaxation time: {t_rh_hunt:} ± {sigma_trh_hunt} Myr")

In [ ]:
np.nanmax(cluster_data['mass'])

In [ ]:
(t_rh_hunt/((10**(fit_params_median['loga'])*u.yr))).to(u.dimensionless_unscaled)

In [ ]:
t_seg_paper_upper = 1.589*u.Msun/(14.71*u.Msun)*(t_rh_hunt+sigma_trh_hunt)
t_seg_paper_lower = 1.589*u.Msun/(14.71*u.Msun)*(t_rh_hunt-sigma_trh_hunt)
t_seg_paper = 1.589*u.Msun/(14.71*u.Msun)*(t_rh_hunt)
sigma_t_seg = (t_seg_paper_upper - t_seg_paper_lower)/2
sigma2_t_seg = 1.589*u.Msun/(14.71*u.Msun)*(sigma_trh_hunt)
t_seg_paper,sigma_t_seg,sigma2_t_seg

In [ ]:
t_seg = np.nanmean(cluster_data['mass'])/np.nanmax(cluster_data['mass'])*t_rh_hunt
print(t_seg)

In [ ]:
np.nanmax(cluster_data['mass'])

In [ ]:
np.nanmean(cluster_data['mass'])

In [ ]:
min_seg = np.nanmean(cluster_data['mass'])*(t_rh_hunt/((10**(fit_params_median['loga'])*u.yr))).decompose()
print(min_seg.to(u.solMass))

In [ ]:
np.nanmin(cluster_data['m1'])

### Cumulative plots

In [ ]:
extracted_centers = [(item[0][0], item[0][1]) for item in centers]
plot_cumulative(data=cluster_data,centers=extracted_centers,R_c=king_results['bayesian_results']['R_c_mean'],R_t=king_results['bayesian_results']['R_t_mean'] ,savefig='../Tex_File/Figures/')

In [ ]:
def plot_cumulative_by_brightness(data, centers, prob_number=[50, 60, 70, 80], brightness_ranges=None, savefig=None, R_c=None, R_t=None, normalize=False, ks=True,paper=False):
    prob_number = np.array(prob_number) / 100
    centers = [SkyCoord(ra=ra, dec=dec, frame='icrs', unit='deg') for ra, dec in centers]
    num_plots = len(prob_number)
    
    if R_c is None:
        R_c = [None] * len(prob_number)
    if R_t is None:
        R_t = [None] * len(prob_number)
    
    if brightness_ranges is None:
        gmag_sorted = np.sort(data['Gmag'])
        quartiles = np.percentile(gmag_sorted, [0, 25, 50, 75, 100])
        brightness_ranges = [(quartiles[i], quartiles[i+1]) for i in range(len(quartiles)-1)]
    
    figsize = (12,8) if len(prob_number) == 2 else (8,8)
    fig, axs = plt.subplots((num_plots + 1) // 2, 2 if num_plots > 1 else 1, figsize=figsize)
    axs = axs.flatten() if num_plots > 1 else [axs]
    
    all_distances = {brightness_range: [] for brightness_range in brightness_ranges}
    
    for i, ax, center, r_c, r_t in zip(prob_number, axs, centers, R_c, R_t):
        selected_data = data[data['probability'] >= i]
        selected_data['d_center'] = angular_separation(selected_data['ra'], selected_data['dec'], center.ra, center.dec).to(u.arcmin)
        x_radius = np.linspace(0, np.nanmax(selected_data['d_center'].value), 400)
        
        cumulative_counts = {brightness_range: [] for brightness_range in brightness_ranges}
        total_counts = {brightness_range: 0 for brightness_range in brightness_ranges}
        
        for (mag_min, mag_max) in brightness_ranges:
            bright_data = selected_data[(selected_data['Gmag'] > mag_min) & (selected_data['Gmag'] <= mag_max)]
            total_counts[(mag_min, mag_max)] = len(bright_data)
            all_distances[(mag_min, mag_max)].extend(bright_data['d_center'].value)
        
        for r in x_radius:
            for (mag_min, mag_max) in brightness_ranges:
                count = len(selected_data[(selected_data['d_center'] <= r*u.arcmin) & 
                                          (selected_data['Gmag'] > mag_min) & 
                                          (selected_data['Gmag'] <= mag_max)])
                cumulative_counts[(mag_min, mag_max)].append(count)
        
        for (mag_min, mag_max), counts in cumulative_counts.items():
            if total_counts[(mag_min, mag_max)] > 0:
                normalized_counts = np.array(counts) / total_counts[(mag_min, mag_max)] if normalize else counts
                label = rf'$G_{{mag}}$: {mag_min.value:.2f} to {mag_max:.2f}'
                ax.plot(x_radius, normalized_counts, label=label)
        
        if r_c is not None and r_c.value:
            ax.axvline(r_c.value, color='green', label=rf'$R_c = {r_c.value:.2f}$ {r_c.unit}', linestyle='-.', linewidth=2, alpha=0.8,zorder=-1)
        if r_t is not None and r_t.value:
            ax.axvline(r_t.value, color='red', label=rf'$R_t = {r_t.value:.2f}$ {r_t.unit}', linestyle='-.', linewidth=2, alpha=0.8,zorder=-1)
        
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.set_xlabel('Radius [arcmin]',fontsize=16)
        ax.set_ylabel('Normalized cumulative count' if normalize else 'Cumulative count',fontsize=16)
        ax.legend()
        if not paper:
            ax.set_title(f"{int(i * 100)}% probability members")
    
    if ks:
        print("K-S test results:")
        for ((min_i, max_i), dist_i) in all_distances.items():
            for ((min_j, max_j), dist_j) in all_distances.items():
                if (min_i, max_i) < (min_j, max_j):  # Avoid comparing the same range and ensure unique pairs
                    ks_stat, ks_pvalue = ks_2samp(dist_i, dist_j)
                    print(f"Between brightness range {min_i:.2f}, {max_i:.2f} and {min_j:.2f}, {max_j:.2f}: KS-statistic={ks_stat:.2f}, p-value={ks_pvalue:.2f}")

    if savefig and paper:
        plt.savefig(savefig + 'cumulative_by_brightness_paper.pdf', bbox_inches='tight')
    else:
        plt.savefig(savefig + 'cumulative_by_brightness.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
plot_cumulative_by_brightness(data=cluster_data,prob_number=[60],centers=extracted_centers,
                              R_c=king_results['bayesian_results']['R_c_mean'],R_t=king_results['bayesian_results']['R_t_mean'],
                              savefig='../Tex_File/Figures/',normalize=True,ks=True,paper=True)

In [ ]:
def plot_cumulative_by_mass_and_type(data, centers, prob_number=[70], savefig=None, normalize=False, ks=True,m_seg=None):
    prob_number = np.array(prob_number) / 100
    centers = [SkyCoord(ra=ra, dec=dec, frame='icrs', unit='deg') for ra, dec in centers]
    # Filter data based on probability
    data = data[data['probability'] >= prob_number[0]]
    if m_seg is not None:
        data = data[data['mass'] <= m_seg]
    # Remove NaN values and sort mass data
    mass_sorted = np.sort(data['mass'][~np.isnan(data['mass'])])
    quartiles = np.percentile(mass_sorted, [0, 25, 50, 75, 100])
    mass_ranges = [(quartiles[i], quartiles[i+1]) for i in range(4)]
    print(len(data[~np.isnan(data['mass'])]),np.nanmean(data['mass']))
    # Set up figure
    figsize = (18, 6)
    fig, axs = plt.subplots(1, 3, figsize=figsize, squeeze=False,sharex=True)
    axs = axs.flatten()


    # Calculate angular separation to cluster center
    data['d_center'] = angular_separation(data['ra'], data['dec'], centers[0].ra, centers[0].dec).to(u.arcmin)

    # Define the maximum radius for the x-axis
    max_radius = data['d_center'].max()

    # Split data into single and binary stars
    binary_data = data[data['binar_prob'] >= 0.6]
    single_data = data[data['binar_prob']  < 0.6]

    # Set up datasets for singles and binaries
    datasets = [('Single Stars', single_data), ('Binary Stars', binary_data)]
    ks_results = {}

    # Process each dataset
    for idx, (title, selected_data) in enumerate(datasets):
        ax = axs[idx]
        all_dists = []

        for (mass_min, mass_max) in mass_ranges:
            mask = (selected_data['mass'] >= mass_min) & (selected_data['mass'] < mass_max)
            dists = selected_data[mask]['d_center']
            # Skip the mass range if no data points are present
            if len(dists) == 0:
                continue
            all_dists.append(dists)
            # Compute cumulative distribution
            cumulative_dist = np.array([np.sum(dists <= r) for r in np.linspace(0, max_radius, 400)], dtype=np.float64)
            if normalize:
                max_cumulative_dist = np.max(cumulative_dist)
                if max_cumulative_dist > 0:  # Protect against division by zero
                    cumulative_dist /= max_cumulative_dist
            label = fr'Mass: {mass_min.value:.2f} - {mass_max.value:.2f} $M_\odot$'
            ax.plot(np.linspace(0, max_radius, 400), cumulative_dist, label=label)
        ax.set_title(title,fontsize=16)
        ax.set_xlabel('Radius (arcmin)',fontsize=16)
        if idx == 0:
            ax.set_ylabel('Cumulative distribution' + (' (normalized)' if normalize else ''),fontsize=16)
        ax.legend()

        # Perform KS tests within each dataset if there's more than one mass range
        if ks and len(all_dists) > 1:
            for i in range(len(all_dists)):
                for j in range(i+1, len(all_dists)):
                    if len(all_dists[i]) > 0 and len(all_dists[j]) > 0:
                        ks_stat, ks_pvalue = ks_2samp(all_dists[i], all_dists[j])
                        ks_results[(title, f'Q{i+1} vs Q{j+1}')] = (ks_stat, ks_pvalue)

    # Perform KS test between single and binary stars if both have data
    if len(single_data['d_center']) > 0 and len(binary_data['d_center']) > 0:
        ks_stat, ks_pvalue = ks_2samp(single_data['d_center'], binary_data['d_center'])
        ks_results[('Single vs Binary Stars', 'Overall')] = (ks_stat, ks_pvalue)

    # Plot for combined single and binary stars
    ax = axs[2]
    single_cum_dist = np.array([np.sum(single_data['d_center'] <= r) for r in np.linspace(0, max_radius, 400)], dtype=np.float64)
    binary_cum_dist = np.array([np.sum(binary_data['d_center'] <= r) for r in np.linspace(0, max_radius, 400)], dtype=np.float64)
    if normalize:
        max_single = np.max(single_cum_dist)
        max_binary = np.max(binary_cum_dist)
        if max_single > 0:
            single_cum_dist /= max_single
        if max_binary > 0:
            binary_cum_dist /= max_binary
    ax.plot(np.linspace(0, max_radius, 400), single_cum_dist, label='Single Stars')
    ax.plot(np.linspace(0, max_radius, 400), binary_cum_dist, label='Binary Stars')
    ax.set_title('Single vs Binary Stars',fontsize=16)
    ax.set_xlabel('Radius (arcmin)',fontsize=16)
    ax.legend()

    # Setting minor ticks for the x-axis and y-axis
    for idx, ax in enumerate(axs):
        if idx != 0:
            ax.set_yticks([])
        if idx == 0:
            ax.yaxis.set_major_locator(ticker.AutoLocator())  # Auto-locate y-axis ticks
            ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto-locate minor ticks
        ax.xaxis.set_major_locator(ticker.AutoLocator())  # Auto-locate x-axis ticks
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())  # Auto-locate minor tick
    plt.subplots_adjust(wspace=0, hspace=0)
    # Save the figure if a save path is provided
    if savefig:
        if m_seg is not None:
            plt.savefig(savefig + 'cumulative_by_mass_and_type_mseg.pdf', bbox_inches='tight')
        else:
            plt.savefig(savefig + 'cumulative_by_mass_and_type.pdf', bbox_inches='tight')
    plt.show()

    # Print KS test results
    print("K-S test results:")
    for key, (ks_stat, ks_pvalue) in ks_results.items():
        print(f"{key[0]} - {key[1]}: KS-stat={ks_stat:.2f}, p-value={ks_pvalue:.2f}")

In [ ]:
plot_cumulative_by_mass_and_type(data=cluster_data,prob_number=[60],centers=extracted_centers,savefig='../Tex_File/Figures/',normalize=True,ks=True)
#plot_cumulative_by_mass_and_type(data=data_gaia[cluster],prob_number=[70],centers=extracted_centers,R_c=king_results['bayesian_results']['R_c_mean'],savefig='../Tex_File/Figures/',normalize=True,ks=True)

In [ ]:
cluster_data['m1'].max()

In [ ]:
cluster_data['mass'].unit = u.Msun 

In [ ]:
cluster_data['m1'].mean()

In [ ]:
cluster_data[(cluster_data['m1'] <= min_seg)]['m1'].mean()

In [ ]:
min_seg

In [ ]:
plot_cumulative_by_mass_and_type(data=cluster_data,prob_number=[60]
                                 ,centers=extracted_centers,savefig='../Tex_File/Figures/'
                                 ,normalize=True,ks=True
                                 ,m_seg=min_seg.to(u.Msun))

In [ ]:
cluster_data['Q'] = cluster_data['J_H'] - 1.55 * cluster_data['H_K']

In [ ]:
np.sum(~np.isnan(cluster_data['Q']))

In [ ]:
plt.hist(cluster_data[cluster_data['Q'] <= -0.05*u.mag]['Q'],bins='auto')

In [ ]:
np.nanstd(cluster_data[cluster_data['Q'] <= -0.05*u.mag]['Q']),np.nanmean(cluster_data[cluster_data['Q'] <= -0.05*u.mag]['Q']),np.nanmedian(cluster_data[cluster_data['Q'] <= -0.05*u.mag]['Q'])

In [ ]:
(len(cluster_data[cluster_data['Q'] <= -0.05*u.mag]['Q']))/len(cluster_data[pms | nopms])

In [ ]:
import scipy.stats.distributions as dist

def bayesian_credible_interval(k, n, confidence_level=0.95):
    """
    Calculate the Bayesian credible interval for a binomial proportion.

    Parameters:
    - k: int, number of successes (e.g., number of YSOs).
    - n: int, number of trials (e.g., total number of cluster members).
    - confidence_level: float, desired confidence level (default is 0.95 for a 95% credible interval).

    Returns:
    - p_lower: float, lower bound of the credible interval.
    - p_upper: float, upper bound of the credible interval.
    """
    # Calculate the lower and upper bounds of the credible interval
    p_lower = dist.beta.ppf((1 - confidence_level) / 2., k + 1, n - k + 1)
    p_upper = dist.beta.ppf(1 - (1 - confidence_level) / 2., k + 1, n - k + 1)

    return p_lower, p_upper

# Example usage
k = (len(cluster_data[cluster_data['Q'] <= -0.05*u.mag]['Q']))   # Number of successes (e.g., number of YSOs)
n = len(cluster_data[pms | nopms])  # Number of trials (e.g., total number of cluster members)
confidence_level = 0.95  # 95% credible interval

p_lower, p_upper = bayesian_credible_interval(k, n, confidence_level)

print(f"Bayesian credible interval: [{p_lower}, {p_upper}]")

In [ ]:
import scipy.stats as stats

# Data
N_cl = len(cluster_data[pms | nopms])  # Total number of trials (e.g., total stars in the cluster)
N_YSO = (len(cluster_data[cluster_data['Q'] <= -0.05*u.mag]['Q']))  # Number of successes (e.g., YSOs identified)

# Define the prior parameters (Uniform prior: Beta(1, 1))
alpha_prior = 1
beta_prior = 1

# Posterior parameters
alpha_post = N_YSO + alpha_prior
beta_post = N_cl - N_YSO + beta_prior

# Calculate the 95% credible interval
credible_interval = stats.beta.interval(0.95, alpha_post, beta_post)

# Mean of the posterior distribution
mean_post = stats.beta.mean(alpha_post, beta_post)

credible_interval, mean_post

In [ ]:
#### Parallax observed vs Parallax corrected

fig_pp, ax_pp = plt.subplots(2,2, layout='tight',figsize=(12,8))

pp_sc_gmag = ax_pp[0,0].scatter(cluster_data['parallax_observed'],cluster_data['parallax'],1,c=cluster_data['Gmag'].data,alpha=0.9,cmap=sns.diverging_palette(359,230, l=65, center="dark", as_cmap=True))
ax_pp[0,0].plot(ax_pp[0,0].get_xlim(), ax_pp[0,0].get_ylim(),'m--',alpha=0.5)
fig_pp.colorbar(pp_sc_gmag,ax=ax_pp[0,0]).set_label(r'G magnitud [{}]'.format(cluster_data['Gmag'].unit))
ax_pp[0,0].set(xlabel=r'Observed Parallax $\varpi_{{obs}}$ [{}]'.format(cluster_data['parallax_observed'].unit),ylabel=r'Corrected Parallax $\varpi$ [{}]'.format(cluster_data['Gmag'].unit))
ax_pp[0,0].xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_pp[0,0].yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_pp[0,0].tick_params(axis='both', which='both', direction='in')

pp_sc_nu = ax_pp[0,1].scatter(cluster_data['parallax_observed'],cluster_data['parallax'],1,c=cluster_data['nu_eff_used_in_astrometry'].data,alpha=0.9,cmap=sns.diverging_palette(359,230, l=65, center="dark", as_cmap=True))
ax_pp[0,1].plot(ax_pp[0,1].get_xlim(), ax_pp[0,1].get_ylim(),'m--',alpha=0.5)
fig_pp.colorbar(pp_sc_nu,ax=ax_pp[0,1]).set_label(r'$\nu_{{eff}}$ [{}]'.format((cluster_data['nu_eff_used_in_astrometry'].unit)))
ax_pp[0,1].set(xlabel=r'Observed Parallax $\varpi_{{obs}}$ [{}]'.format(cluster_data['parallax_observed'].unit),ylabel=r'Corrected Parallax $\varpi$ [{}]'.format(cluster_data['nu_eff_used_in_astrometry'].unit))
ax_pp[0,1].xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_pp[0,1].yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_pp[0,1].tick_params(axis='both', which='both', direction='in')

pp_sc_ps = ax_pp[1,0].scatter(cluster_data['parallax_observed'],cluster_data['parallax'],1,c=cluster_data['pseudocolour'].data,alpha=0.9,cmap=sns.diverging_palette(359,230, l=65, center="dark", as_cmap=True))
ax_pp[1,0].plot(ax_pp[1,0].get_xlim(), ax_pp[1,0].get_ylim(),'m--',alpha=0.5)
fig_pp.colorbar(pp_sc_ps,ax=ax_pp[1,0]).set_label(r'Pseudolor [{}]'.format(cluster_data['pseudocolour'].unit))
ax_pp[1,0].set(xlabel=r'Observed Parallax $\varpi_{{obs}}$ [{}]'.format(cluster_data['parallax_observed'].unit),ylabel=r'Corrected Parallax $\varpi$ [{}]'.format(cluster_data['pseudocolour'].unit))
ax_pp[1,0].xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_pp[1,0].yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_pp[1,0].tick_params(axis='both', which='both', direction='in')

pp_sc_lat = ax_pp[1,1].scatter(cluster_data['parallax_observed'],cluster_data['parallax'],1,c=cluster_data['ecl_lat'].data,alpha=0.9,cmap=sns.diverging_palette(359,230, l=65, center="dark", as_cmap=True))
ax_pp[1,1].plot(ax_pp[1,1].get_xlim(), ax_pp[1,1].get_ylim(),'m--',alpha=0.5)
fig_pp.colorbar(pp_sc_lat,ax=ax_pp[1,1]).set_label(r'Ecliptic Latitude [{}]'.format(cluster_data['ecl_lat'].unit))
ax_pp[1,1].set(xlabel=r'Observed Parallax $\varpi_{{obs}}$ [{}]'.format(cluster_data['parallax_observed'].unit),ylabel=r'Corrected Parallax $\varpi$ [{}]'.format(cluster_data['ecl_lat'].unit))
ax_pp[1,1].xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_pp[1,1].yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_pp[1,1].tick_params(axis='both', which='both', direction='in')

fig_pp.suptitle('Corrected parallax and Observed parallax with different parameters')
fig_pp.savefig('../Tex_File/Figures/corr_obs_parallax.pdf');

### Histogram of the Angular distance between 2MASS and DR3

In [ ]:
### Angular distance

fig_cm, ax_cm = plt.subplots(1,1, layout='tight',figsize=(5,5),subplot_kw={'adjustable':'box'})
ax_cm.hist(cluster_data['angular_distance'].data,bins='auto')
ax_cm.set(xlabel=r'Angular Distance GAIA - 2MASS [{}]'.format(cluster_data['angular_distance'].unit),ylabel=r'Number of sources')
ax_cm.xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax_cm.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax_cm.tick_params(axis='both', which='both', direction='in')
fig_cm.suptitle('Angular distance between GAIA DR3 and 2MASS')
fig_cm.savefig('../Tex_File/Figures/angular_separation.pdf');

## Radial velocities model

In [ ]:
np.nanmedian(cluster_data[cluster_data['binar_prob'] <= 0.6]['radial_velocity'])

In [ ]:
np.nanmean(cluster_data[cluster_data['binar_prob'] <= 0.6]['radial_velocity'])

In [ ]:
np.nanstd(cluster_data[cluster_data['binar_prob'] <= 0.6]['radial_velocity'])

In [ ]:
100*np.sum(~np.isnan(cluster_data['radial_velocity']))/len(cluster_data)

In [ ]:
# Check for missing rv_amplitude_robust values
if hasattr(cluster_data['rv_amplitude_robust'], 'mask'):
    condition_missing_amp = cluster_data['rv_amplitude_robust'].mask
else:
    condition_missing_amp = np.isnan(cluster_data['rv_amplitude_robust'])

# Create the plot
fig, ax = plt.subplots(layout='tight',figsize=(8,7))

# Group 1
ax.errorbar(cluster_data[condition_missing_amp]['radial_velocity'], cluster_data[condition_missing_amp]['Gmag'],
            xerr=cluster_data[condition_missing_amp]['radial_velocity_error'],
            fmt='o', ecolor='gray', mec='black', mfc='yellow', label='Sources without amplitude')

# Group 2
ax.errorbar(cluster_data[~condition_missing_amp]['radial_velocity'], cluster_data[~condition_missing_amp]['Gmag'],
            xerr=cluster_data[~condition_missing_amp]['radial_velocity_error'],
            fmt='none', ecolor='gray', mec='black', mfc='lightblue',zorder=-1)

scatter = ax.scatter(cluster_data[~condition_missing_amp]['radial_velocity'], 
                     cluster_data[~condition_missing_amp]['Gmag'],
                     c=cluster_data[~condition_missing_amp]['rv_amplitude_robust'].value,
                     s=60, edgecolor='black', cmap='coolwarm', label='Sources with amplitude')
# Color bar
cbar = fig.colorbar(scatter)
cbar.set_label(f'Amplitude [{cluster_data['rv_amplitude_robust'].unit}]',fontsize=16)

# Additional plot settings
ax.invert_yaxis()
ax.set_xlabel(f'Radial velocity [{cluster_data['radial_velocity'].unit}]',fontsize=16)
ax.set_ylabel(rf'$G_\mathrm{{mag}} $ [{cluster_data['Gmag'].unit}]',fontsize=16)
ax.legend()
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.tick_params(axis='both', which='both', direction='in',labelsize=14)
plt.show()
fig.savefig('../Tex_File/Figures/radial_velocity_amplitude.pdf',bbox_inches='tight')


In [ ]:
table_info = Simbad.list_columns('mesSpT')

In [ ]:
table_info

In [ ]:
from astroquery.simbad import Simbad

# Define the corrected query
query = """
SELECT main_id AS "Main Identifier", 
       basic.otype AS "Object Type", 
       ra AS "Right Ascension", 
       dec AS "Declination", 
       ids.ids AS "All Identifiers",
       h_link.membership AS "Membership Probability",
       cluster_table.id AS "Parent Cluster",
       otypedef.description AS "Object Type Definitions",
       mesSpT.sptype AS "Spectral Type"
FROM ident AS cluster_table
JOIN h_link ON cluster_table.oidref = h_link.parent
JOIN basic ON h_link.child = basic.oid
JOIN ids ON basic.oid = ids.oidref
LEFT JOIN otypedef ON basic.otype = otypedef.otype
LEFT JOIN mesSpT ON basic.oid = mesSpT.oidref
WHERE cluster_table.id = 'NGC 6383';
"""

# Execute the query using SIMBAD's TAP service
result_table = Simbad.query_tap(query)

# Print the results to check
print(result_table)

In [ ]:
# Function to find the preferred designation
def find_preferred_designation(all_ids, main_id):
    # Split the concatenated identifiers by '|'
    identifiers = all_ids.split('|')
    
    # Try to find identifiers in preferred order
    for prefix in ['Gaia DR3', 'Gaia', '2MASS']:
        for identifier in identifiers:
            if identifier.startswith(prefix):
                return identifier
    
    # If no preferred identifier is found, return the main identifier
    return main_id

# Apply the function to each row in the result_table
result_table['designation'] = [find_preferred_designation(row['All Identifiers'], row['Main Identifier']) for row in result_table]
#result_table['designation_dr2'] = [find_preferred_designation(row['All Identifiers'], row['Main Identifier']) for row in result_table]

In [ ]:
for i in result_table['designation','All Identifiers','Main Identifier']:
    if not i['designation'].startswith('Gaia DR3'):
        print(i['designation'])

In [ ]:
# Iterate through the table to update specific designation from Gaia DR2 to Gaia DR3
for i in range(len(result_table)):
    if 'Gaia DR2 4054565469499817728' in result_table['designation'][i]:
        # Replace 'Gaia DR2' with 'Gaia DR3'
        result_table['designation'][i] = result_table['designation'][i].replace('Gaia DR2', 'Gaia DR3')
        print(f"Updated row {i}: {result_table['designation'][i]}")


In [ ]:
result_table = unique(result_table,keys='designation')

In [ ]:
ct2020 = unique(QTable.read('../cantat_gaudin_2020.fit'),keys='GaiaDR2')
he2022 = unique(QTable.read('../he_2022.fit'),keys='GaiaEDR3')
jaehnig = unique(QTable.read('../Jaehnig_2021.fit'),keys='Gaia')

In [ ]:
from astropy.table import join, setdiff,unique

# Assuming 'he2022' and 'data_gaia' are your QTable objects and the columns 'GaiaEDR3' and 'source_id' contain the IDs

# Rename 'GaiaEDR3' in he2022 to 'source_id' to match data_gaia for joining
#he2022.rename_column('GaiaEDR3', 'source_id')

# Inner Join (common elements)
common_sources_he2022 = join(he2022, data_gaia[cluster], keys='source_id', join_type='inner')

# Set Difference (elements unique to he2022)
unique_he2022 = setdiff(he2022, data_gaia[cluster], keys='source_id')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_he2022 = setdiff(data_gaia[cluster], he2022, keys='source_id')

# Print results
print("Common Sources:", len(common_sources_he2022))
print("Unique to HE2022:", len(unique_he2022))
print("Unique to Data Gaia:", len(unique_data_gaia_he2022))

In [ ]:
#aehnig.rename_column('Gaia', 'source_id')

common_sources_jaehnig = join(jaehnig, data_gaia[cluster], keys='source_id', join_type='inner')

# Set Difference (elements unique to he2022)
unique_jaehnig = setdiff(jaehnig, data_gaia[cluster], keys='source_id')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_jaehnig = setdiff(data_gaia[cluster], jaehnig, keys='source_id')

# Print results
print("Common Sources:", len(common_sources_jaehnig))
print("Unique to Jaehnig:", len(unique_jaehnig))
print("Unique to Data Gaia:", len(unique_data_gaia_jaehnig))

In [ ]:
#ct2020.rename_column('GaiaDR2', 'source_id')

common_sources_ct = join(ct2020, data_gaia[cluster], keys='source_id', join_type='inner')

# Set Difference (elements unique to he2022)
unique_ct = setdiff(ct2020, data_gaia[cluster], keys='source_id')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_ct = setdiff(data_gaia[cluster], ct2020, keys='source_id')

# Print results
print("Common Sources:", len(common_sources_ct))
print("Unique to CT:", len(unique_ct))
print("Unique to Data Gaia:", len(unique_data_gaia_ct))

In [ ]:
np.unique(result_table['Object Type Definitions'])

In [ ]:
data_gaia['source_id'][0]

In [ ]:
len('4054177651131258240')

In [ ]:
len('405456286245800')

In [ ]:
hunt2024 = QTable.read('/Users/notluquis/Downloads/members.csv')['source_id','mass_50']

In [ ]:
hunt2024

In [ ]:
common_sources_hunt2024 = join(hunt2024, data_gaia[cluster], keys='source_id', join_type='inner')

# Set Difference (elements unique to he2022)
unique_hunt2024 = setdiff(hunt2024, data_gaia[cluster], keys='source_id')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_hunt2024 = setdiff(data_gaia[cluster], hunt2024, keys='source_id')

# Print results
print("Common Sources:", len(common_sources_hunt2024))
print("Unique to CT:", len(unique_hunt2024))
print("Unique to Data Gaia:", len(unique_data_gaia_hunt2024))

In [ ]:
common_sources_simbad = join(result_table, data_gaia[cluster], keys='designation', join_type='inner')

# Set Difference (elements unique to he2022)
unique_simbad = setdiff(result_table, data_gaia[cluster], keys='designation')

# Set Difference (elements unique to data_gaia)
unique_data_gaia_simbad = setdiff(data_gaia[cluster], result_table, keys='designation')

# Print results
print("Common Sources:", len(common_sources_simbad))
print("Unique to CT:", len(unique_simbad))
print("Unique to Data Gaia:", len(unique_data_gaia_simbad))

In [ ]:
np.unique(common_sources_simbad['Object Type Definitions'])

In [ ]:
a = np.unique(common_sources_simbad['Spectral Type'])

In [ ]:
result_table[result_table['Object Type Definitions'] == 'T Tauri Star']

In [ ]:
common_sources_simbad[common_sources_simbad['Object Type Definitions'] == 'T Tauri Star']['designation','Spectral Type','probability','binar_prob','mass','pms_sagitta','av_sagitta']

In [ ]:
common_sources_simbad[common_sources_simbad['Spectral Type'] == a[-1]]

In [ ]:
common_sources_simbad[common_sources_simbad['Spectral Type'] == a[3]]

In [ ]:
common_sources_simbad[common_sources_simbad['Object Type Definitions'] == 'Eclipsing Binary']

In [ ]:
cluster_data.colnames

In [ ]:
# Assuming `cluster_data` is already a QTable
# Select only the specified columns
selected_columns = ['source_id', 'ra', 'dec', 'ra_error', 'dec_error', 'pmra', 'pmdec', 'pmra_error',
                    'pmdec_error', 'Gmag', 'G_RPmag', 'BP_RP', 'bp_g', 'fidelity_v2', 'G_BPmag', 
                    'radial_velocity', 'tmass_designation', 'tmass_oid', 'j_m', 'j_msigcom', 
                    'h_m', 'h_msigcom', 'ks_m', 'ks_msigcom', 'r_med_geo', 'r_lo_geo', 
                    'r_hi_geo', 'r_med_photogeo', 'r_lo_photogeo', 'r_hi_photogeo', 'e_Gmag', 
                    'e_G_BPmag', 'e_G_RPmag', 'e_BP_RP', 'e_J_H', 'e_RP_J', 'e_H_K', 
                    'e_BP_J', 'RP_J', 'H_K', 'J_K', 'J_H', 'BP_J', 'cluster', 
                    'probability_hdbscan', 'probability_times', 'probability', 
                    'outlier_score', 'cluster_hdbscan', 'av_sagitta', 'pms_sagitta', 
                    'age', 'projected_velocity', 'd_center', 'm1', 'm1_std', 
                    'm2', 'm2_std', 'binar_prob', 'mass', 'mass_std']

# Select the columns from the QTable
filtered_cluster_data = cluster_data[selected_columns]

# Save the QTable to an ECSV file
filtered_cluster_data.write('cluster_data_final.ecsv', format='ascii.ecsv', overwrite=True)

## Other stuff 

In [ ]:
def cmd(data,cluster_element,isocrone_df=[0],z_array=[0]*4,modulus_distance=0,absorption=0,age_array=[0]*4,prob_number=[50,60,70,80], save_path='../Tex_File/Figures/HR_ngc6383.pdf',id=0):
    prob_number = np.array(prob_number) / 100
    num_plots = len(prob_number)
    # Setup subplots
    fig, axs = plt.subplots((num_plots + 1) // 2, 2 if num_plots > 1 else 1, figsize=(13, 9*num_plots))
    axs = axs.flatten() if num_plots > 1 else [axs]
    data_cluster = data[cluster_element]
    for i, ax,z,a in zip(prob_number, axs,z_array,age_array):
        # Filter using query or boolean indexing
        filtered_isochrone = isocrone_df.query(f"logAge == {a} and Zini == {z}")
        probability_th = (data_cluster['probability'] >= i)
        selected_data = data_cluster[probability_th]
        ax.scatter(data['BP_RP'][~cluster_element],data['Gmag'][~cluster_element],alpha=0.05,c='gray')
        ax.scatter(data_cluster['BP_RP'][~probability_th],data_cluster['Gmag'][~probability_th],alpha=0.05,c='gray',edgecolors='black')
        sc_cmd = ax.scatter(selected_data['BP_RP'],selected_data['Gmag'],c=selected_data['probability'],cmap=sns.color_palette("flare", as_cmap=True),s=14)
        fig.colorbar(sc_cmd).set_label(r'Probabilities')
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator()),ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.tick_params(axis='both', which='both', direction='in')
        ax.set(xlabel=(r'Color $(BP - RP)$ [{}]'.format(data['BP_RP'].unit)),ylabel=(r'Magnitude $G$ [{}]'.format(data['Gmag'].unit)))
        x,y = ax.get_xlim(),ax.get_ylim()
        ax.scatter(filtered_isochrone['BP_RP']+absorption, filtered_isochrone['Gmag']+modulus_distance, s=5, c='blue', label=f'Age: 10^{a} years, Z={z}')
        ax.set(ylim=y,xlim=x)
        ax.invert_yaxis()
    fig.savefig('../Tex_File/Figures/HR_ngc6383.pdf');
cmd(data_gaia,cluster,isocrone_df=all_isochrones_DR3,prob_number=[70])#,z_array=[0.0201]*4,age_array=[6.5]*4,modulus_distance=10.47,absorption=0.156)

def cmd(data, cluster_element, isochrone_df, modulus_distance=0, absorption=0, prob_number=[50,60,70,80], save_path='../Tex_File/Figures/HR_ngc6383.pdf', id=None, z_array=None, age_array=None):
    prob_number = np.array(prob_number) / 100
    num_plots = len(prob_number)
    # Setup subplots
    fig, axs = plt.subplots((num_plots + 1) // 2, 2 if num_plots > 1 else 1, figsize=(13, 7 * num_plots // 2.5))
    axs = axs.flatten() if num_plots > 1 else [axs]
    data_cluster = data[cluster_element]

    # If an ID is provided, filter using ID, ignoring z_array and age_array
    if id is not None:
        filtered_isochrone = isochrone_df[isochrone_df['isochrone_id'] == id]
        # Plot for a single isochrone ID
        plot_isochrone_data(axs, data, cluster_element, filtered_isochrone, modulus_distance, absorption, prob_number)
    else:
        # Proceed with z_array and age_array if no ID is provided
        if z_array is not None and age_array is not None:
            for i, ax, z, a in zip(prob_number, axs, z_array, age_array):
                filtered_isochrone = isochrone_df.query(f"logAge == {a} and Zini == {z}")
                plot_isochrone_data(ax, data, cluster_element, filtered_isochrone, modulus_distance, absorption, [i])
        else:
            raise ValueError("Either an ID or both z_array and age_array must be provided.")

    fig.savefig(save_path)

def plot_isochrone_data(axs, data, cluster_element, filtered_isochrone, modulus_distance, absorption, prob_numbers):
    for i, ax in zip(prob_numbers, axs):
        probability_th = data[cluster_element]['probability'] >= i
        selected_data = data[cluster_element][probability_th]
        ax.scatter(data['BP_RP'][~cluster_element], data['Gmag'][~cluster_element], alpha=0.05, c='gray')
        ax.scatter(selected_data['BP_RP'], selected_data['Gmag'], c=selected_data['probability'], cmap=sns.color_palette("flare", as_cmap=True), s=14, label=f'Prob >= {i*100}%')
        if not filtered_isochrone.empty:
                ax.scatter(filtered_isochrone['BP_RP'] + absorption, filtered_isochrone['Gmag'] + modulus_distance, s=5, c='blue', label='Isochrone')
        ax.legend()
        ax.invert_yaxis()
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(axis='both', which='both', direction='in')
        ax.set_xlabel('Color $(BP - RP)$')
        ax.set_ylabel('Magnitude $G$')

# ASTECA

In [ ]:
# ### =============================================================================
# ### #                                   ASTECA formatting.
# #### =============================================================================

write_data = cluster_data & (data_gaia['Gmag'] <= 18*u.mag)]
pms = (write_data['pms_sagitta'] >= 0.6)
nopms = (write_data['pms_sagitta'] < 0.6)
filedir = '../ASteCA/input/'
#for i in ['2mass','dr3']:
for i in ['dr3']:
    if i == '2mass':
        write = write_data['designation','ra','dec','parallax','parallax_error','pmra','pmra_error','pmdec','pmdec_error','j_m','RP_J','j_msigcom','e_BP_RP']
        write.rename_columns(['j_m','j_msigcom'],['Jmag','e_Jmag'])
        ascii.write(write,f'{filedir}NGC_6383_{i}_all_18mag.csv',format='csv',overwrite=True,delimiter=',')
        #ascii.write(write[pms],f'{filedir}NGC_6383_{i}_pms.csv',format='csv',overwrite=True,delimiter=',')
        #scii.write(write[nopms],f'{filedir}NGC_6383_{i}_nopms.csv',format='csv',overwrite=True,delimiter=',')
    if i == "dr3":
        write = write_data['designation','ra','dec','parallax','parallax_error','pmra','pmra_error','pmdec','pmdec_error','Gmag','BP_RP','e_Gmag','e_BP_RP']
        print(len(write))
        ascii.write(write,f'{filedir}NGC_6383_{i}_all_18mag.csv',format='csv',overwrite=True,delimiter=',')
        #ascii.write(write[pms],f'{filedir}NGC_6383_{i}_pms.csv',format='csv',overwrite=True,delimiter=',')
        #ascii.write(write[nopms],f'{filedir}NGC_6383_{i}_nopms.csv',format='csv',overwrite=True,delimiter=',')

## Outlier score

In [ ]:
import pandas as pd

In [ ]:
hunt24 = pd.read_csv('/Users/notluquis/Downloads/gogo/members.csv')

In [ ]:
ngc6383 = hunt24[hunt24['name'] == "NGC_6383"]

In [ ]:
ngc6383

In [ ]:
masses = ngc6383['mass_50']
masses = masses[masses > 0]
n, bins = np.histogram(masses, np.arange(np.min(masses), np.max(masses), 0.5))

# Calcula el centro de cada bin
M = 0.5 * (bins[1:] + bins[:-1])
dM = np.diff(bins)
dN = n

# Para dN/dM, divide el número de objetos por el ancho del bin
dN_dM = dN / dM

# Calcula los errores de Poisson como la raíz cuadrada de los conteos en cada bin
errors = np.sqrt(n) / dM

fig, ax = plt.subplots(1, 1, layout='tight', figsize=(7, 3))

# Crear el gráfico con barras de error
ax.errorbar(M, dN_dM, yerr=errors, marker='o', linestyle='None', capsize=5)

# Añadir títulos y etiquetas
ax.set_xlabel('Masa (M)')
ax.set_ylabel('dN/dM')
ax.set_yscale('log')
ax.set_xscale('log')
plt.show()

In [ ]:
cluster_criteria = hunt24['name'] == 'NGC_6383'
ngc6383 = hunt24[cluster_criteria]

In [ ]:
ngc6383['mass_2.5'].sum()

In [ ]:
ngc6383['mass_16'].sum()

In [ ]:
ngc6383['mass_50'].sum()

In [ ]:
ngc6383['mass_84'].sum()

In [ ]:
ngc6383['mass_97.5'].sum()

In [ ]:
gg = pd.read_csv('/Users/notluquis/Downloads/clusters.csv')

In [ ]:
pp = gg[gg['name'] == 'NGC_6383']

In [ ]:
pd.set_option('display.max_columns', None)